In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/.format_version
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/.storage_alignment
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data.pkl
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/version
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/byteorder
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/.data/serialization_id
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data/437
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data/515
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data/248
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data/625
/kaggle/input/models/srishtipandey14/v4-full-best-pt/pytorch/default/1/best/data/7
/kaggle/input/models/srishtipandey14

In [2]:
import glob
import yaml
from collections import Counter

dataset_path = '/kaggle/input/datasets/srishtipandey14/unified-dataset-sih'
yaml_path = f'{dataset_path}/classes.yaml'

# Load class names
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)
    class_names = config.get('names', [])
    if isinstance(class_names, dict):
        class_names = [class_names[i] for i in range(len(class_names))]

# Extract stats for train and val splits
for split in ['train', 'val']:
    label_files = glob.glob(f'{dataset_path}/labels/{split}/*.txt')
    class_counts = Counter()
    
    for label_path in label_files:
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

    print(f"📊 --- {split.upper()} SPLIT STATS ---")
    print(f"Total Images/Labels: {len(label_files)}")
    
    if label_files:
        print("Instances per class:")
        for class_id in sorted(class_counts.keys()):
            name = class_names[class_id] if class_id < len(class_names) else f"ID {class_id}"
            print(f"  - {name}: {class_counts[class_id]}")
    print("\n")

📊 --- TRAIN SPLIT STATS ---
Total Images/Labels: 19033
Instances per class:
  - person: 64888
  - fire: 11809
  - smoke: 9517
  - floodwater: 2520
  - structural_damage: 5976
  - landslide: 383
  - exposed_wire: 520


📊 --- VAL SPLIT STATS ---
Total Images/Labels: 4737
Instances per class:
  - person: 18702
  - fire: 2876
  - smoke: 2335
  - floodwater: 567
  - structural_damage: 1448
  - landslide: 92
  - exposed_wire: 114




In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 869.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 5.7 MB/s eta 0:00:00


In [4]:
import os

print("📂 Folders found inside /kaggle/input:")
for root, dirs, files in os.walk('/kaggle/input'):
    # Show directories that might contain images or labels
    if any(k in root.lower() for k in ['flood', 'human', 'earthquake']):
        print(root)
        break

📂 Folders found inside /kaggle/input:
/kaggle/input/datasets/constantinwerner/human-detection-dataset


In [5]:
import os
import glob
import yaml
from ultralytics import YOLO

print("⚙️ Step 1: Recursively Locating Datasets & Model...")

# 1. Original YAML
original_yaml = '/kaggle/input/datasets/srishtipandey14/unified-dataset-sih/classes.yaml'
if not os.path.exists(original_yaml):
    original_yaml = next((p for p in glob.glob('/kaggle/input/**/*.yaml', recursive=True) if 'unified' in p.lower()), None)

# 2. Backbone Model
p1_weights = '/kaggle/input/models/srishtipandey14/v4-best-pt/pytorch/default/1/best (1).pt'
if not os.path.exists(p1_weights):
    p1_weights = next((p for p in glob.glob('/kaggle/input/**/*.pt', recursive=True) if 'v4' in p.lower() and 'best' in p.lower()), None)

# 3. Flood/Human Dataset folder search across all subdirectories
candidate_dirs = []
for root, dirs, files in os.walk('/kaggle/input'):
    # Look for train image folders inside flood or human directories
    if ('flood' in root.lower() or 'human' in root.lower()) and ('train' in root.lower() or 'images' in root.lower()):
        candidate_dirs.append(root)

print(f"✅ Original YAML: {original_yaml}")
print(f"✅ Backbone Model: {p1_weights}")
print(f"🔍 Candidate Flood/Human directories found: {len(candidate_dirs)}")
for d in candidate_dirs[:5]:
    print("   ->", d)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
⚙️ Step 1: Recursively Locating Datasets & Model...
✅ Original YAML: /kaggle/input/datasets/srishtipandey14/unified-dataset-sih/classes.yaml
✅ Backbone Model: /kaggle/input/models/srishtipandey14/v4-best-pt/pytorch/default/1/best (1).pt
🔍 Candidate Flood/Human directories found: 4
   -> /kaggle/input/datasets/sriramm932/detecting-humans-in-floods-4nmv0/valid/images
   -> /kaggle/input/datasets/sriramm932/detecting-humans-in-floods-4nmv0/train
   -> /kaggle/input/datasets/sriramm932/detecting-humans-in-floods-4nmv0/train/labels
   -> /kaggle/input/datasets/sriramm932/detecting-humans-in-floods-4nmv0/train/images


In [6]:
import os
import yaml
from ultralytics import YOLO

# ==============================================================================
# 1. CREATE MERGED YAML (Guaranteed to exist on clean commit runs)
# ==============================================================================
print("⚙️ Step 1: Writing merged YAML configuration...")

original_yaml = '/kaggle/input/datasets/srishtipandey14/unified-dataset-sih/classes.yaml'
new_dataset_dir = '/kaggle/input/datasets/sriramm932/detecting-humans-in-floods-4nmv0'
p1_weights = '/kaggle/input/models/srishtipandey14/v4-best-pt/pytorch/default/1/best (1).pt'

with open(original_yaml, 'r') as f:
    config = yaml.safe_load(f)

# Re-map train/val to include both original hazards and flood humans
original_img_path = os.path.dirname(original_yaml)
config['path'] = '/kaggle/input'
config['train'] = [
    os.path.join(original_img_path, 'images/train'),
    os.path.join(new_dataset_dir, 'train/images')
]
config['val'] = [
    os.path.join(original_img_path, 'images/val'),
    os.path.join(new_dataset_dir, 'valid/images')
]

merged_yaml_path = '/kaggle/working/merged_phase2.yaml'
with open(merged_yaml_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)

print(f"✅ Created configuration at: {merged_yaml_path}")

# ==============================================================================
# 2. RUN HIGH-SPEED CLEAN TRAINING
# ==============================================================================
print("\n🚀 Step 2: Launching clean v5 fine-tuning...")

v5_model = YOLO(p1_weights)

v5_model.train(
    data=merged_yaml_path,
    epochs=40,
    imgsz=640,
    batch=32,            # Fast batch size to avoid CPU queueing
    workers=2,           # Prevents Kaggle vCPU thread locks
    freeze=0,
    optimizer='AdamW',
    lr0=0.0001,
    lrf=0.01,
    rect=False,
    
    # Safe scaling without per-batch CPU stall
    multi_scale=False,
    scale=0.5,
    mosaic=1.0,
    
    # Hallucination fixers (disabled)
    copy_paste=0.0,
    mixup=0.0,
    
    close_mosaic=10,
    box=7.5,
    cls=0.5,
    max_det=900,
    cache='ram',
    device=0,
    project='/kaggle/working/models',
    name='v5_final_deployment_ready'
)

⚙️ Step 1: Writing merged YAML configuration...
✅ Created configuration at: /kaggle/working/merged_phase2.yaml

🚀 Step 2: Launching clean v5 fine-tuning...
Ultralytics 8.4.154 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/merged_phase2.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_r

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/40      6.51G       2.25      1.981   0.008792         68        640: 100% ━━━━━━━━━━━━ 644/644 2.4it/s 4:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 81/81 2.9it/s 28.0s
                   all       5126      28613      0.622      0.502       0.51      0.283

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/40      7.29G      2.182      1.859   0.008171         88        640: 100% ━━━━━━━━━━━━ 644/644 2.3it/s 4:34
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 81/81 2.9it/s 27.9s
                   all       5126      28613      0.676      0.545      0.592      0.352

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/40      7.85G      2.158      1.785   0.007829        125        640: 100% ━━━━━━━━━━━━ 644/644 2.3it/s 4:35
                 Class     Images  Instances      Box(

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79ad805124b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  